In [ ]:
!pip install -q transformers accelerate datasets bitsandbytes trl peft tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 13.1 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

In [ ]:
import os
if "COLAB_GPU" in os.environ:
  from google.colab import output
  output.enable_custom_widget_manager()

In [ ]:
import os
from huggingface_hub import notebook_login

# If running in Google Colab
if "COLAB_GPU" in os.environ:
    !huggingface-cli login
# If running locally (Jupyter, VS Code, etc.)
else:
    notebook_login() 


⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: read).
The token `lk` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no 

In [ ]:
dataset = load_dataset("Amod/mental_health_counseling_conversations")
dataset = dataset["train"].train_test_split(test_size=0.30, seed=42)
train_dataset = dataset["train"]
test_dataset = dataset["test"]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [ ]:

# Convert to pandas
train_df = train_dataset.to_pandas()
test_df  = test_dataset.to_pandas()

# Reset index
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)



In [ ]:
def format_prompt(example):
    return {
        "text": (
            f"<|begin_of_text|>\n"
            f"Context: {example['Context']}\n"
            f"Response: {example['Response']}\n"
            f"<|end_of_text|>"
        )
    }


In [ ]:
train_dataset = train_dataset.map(format_prompt)
test_dataset  = test_dataset.map(format_prompt)


Map:   0%|          | 0/2458 [00:00<?, ? examples/s]

Map:   0%|          | 0/1054 [00:00<?, ? examples/s]

In [ ]:

# TOKENIZER  

model_name = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

max_length = 512  # SAME as  LLaMA setup


def tokenize(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
    )
    enc["labels"] = enc["input_ids"].copy()  # causal LM labels (same logic)
    return enc


train_tokenized = train_dataset.map(
    tokenize,
    batched=True,
    remove_columns=train_dataset.column_names
)

test_tokenized  = test_dataset.map(
    tokenize,
    batched=True,
    remove_columns=test_dataset.column_names
)

train_tokenized.set_format("torch")
test_tokenized.set_format("torch")


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Map:   0%|          | 0/2458 [00:00<?, ? examples/s]

Map:   0%|          | 0/1054 [00:00<?, ? examples/s]

In [ ]:
print(train_tokenized[0].keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [ ]:

# QLoRA 4-BIT QUANTIZATION 

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


# LOAD GEMMA MODEL IN 4-BIT (QLoRA)

model = AutoModelForCausalLM.from_pretrained(
    model_name,               # must be: "google/gemma-3-1b-it"
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16    # recommended for Gemma
)


model.config.use_cache = False

model = prepare_model_for_kbit_training(model)


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [ ]:

# LoRA CONFIGURATION 

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",

    # Gemma uses the SAME projection layer names as LLaMA
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

# Apply LoRA to Gemma
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


trainable params: 13,045,760 || all params: 1,012,931,712 || trainable%: 1.2879


In [ ]:

# TRAINING ARGUMENTS 

output_dir = "./qlora-gemma3-mental-health"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=50,

    eval_steps=300,            # evaluate every 300 steps
    eval_strategy="epoch",
    save_strategy="epoch",     # same as LLaMA version

    bf16=True,                 # safe for Gemma-3
    report_to="none",
)


training_args.label_names = ["labels"]


In [ ]:

# DATA COLLATOR + TRAINER 

from transformers import DataCollatorForLanguageModeling, Trainer

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,               #  CAUSAL LM = mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
)


# TRAIN 

trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,2.254700,2.190380
2,1.945800,2.007899
3,1.716500,1.945635


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=462, training_loss=2.0610576192021886, metrics={'train_runtime': 13922.1967, 'train_samples_per_second': 0.53, 'train_steps_per_second': 0.033, 'total_flos': 1.6104913951260672e+16, 'train_loss': 2.0610576192021886, 'epoch': 3.0})

In [ ]:
# Save LoRA adapter + tokenizer
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("✅ QLoRA fine-tuning finished. Adapter saved to:", output_dir)

✅ QLoRA fine-tuning finished. Adapter saved to: ./qlora-gemma3-mental-health


In [ ]:
from huggingface_hub import login
login()

In [ ]:
# #upload login token
# import os
# from huggingface_hub import notebook_login

# # If running in Google Colab
# if "COLAB_GPU" in os.environ:
#     !huggingface-cli login
# # If running locally (Jupyter, VS Code, etc.)
# else:
#     notebook_login() 


In [ ]:
from huggingface_hub import upload_folder

upload_folder(
    folder_path="./qlora-gemma3-mental-health",  #
    repo_id="ZunairAhmad/gemma3-mental-health",
    repo_type="model"
)

print("✔ Uploaded successfully!")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ckpoint-154/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...kpoint-308/tokenizer.json:  75%|#######5  | 25.2MB / 33.4MB            

  ...kpoint-154/tokenizer.json:  50%|#####     | 16.7MB / 33.4MB            

...

  ...ckpoint-462/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...eckpoint-154/scheduler.pt:   1%|1         |  15.0B / 1.47kB            

  ...int-154/training_args.bin:   1%|1         |  62.0B / 5.84kB            

  ...int-308/training_args.bin:   1%|1         |  62.0B / 5.84kB            

✔ Uploaded successfully!
